<a href="https://colab.research.google.com/github/ameesha543/Statistical-Learning-e23095/blob/main/Assignment_3/Question-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Probability Space
*   $\Omega = \{\omega_1, \omega_2, \omega_3, \omega_4\}$.
*   **$\sigma$-algebra**: $\mathfrak{F} = \mathcal{P}(\Omega)$ (the power set), which contains $2^4 = 16$ subsets.
*   **Probability Measure**: <br> $P(\{\omega_1\}) = 0.5$, $P(\{\omega_2\}) = 0.1$, $P(\{\omega_3\}) = 0.2$, $P(\{\omega_4\}) = 0.2$.

### 2. $\sigma$-algebra generated by $Y$

$Y$ partitions $\Omega$ into:
*   $A_0 = \{ \omega : Y(\omega) = 0 \} = \{\omega_1, \omega_2\}$
*   $A_1 = \{ \omega : Y(\omega) = 1 \} = \{\omega_3, \omega_4\}$

So, $\sigma(Y) = \{\emptyset, \Omega, \{\omega_1, \omega_2\}, \{\omega_3, \omega_4\}\}$.

### 3. Why $E[X|Y]$ is $\sigma(Y)$-measurable

By definition, the conditional expectation $E[X|Y]$ is a random variable that is measurable with respect to the information provided by $Y$. This means $E[X|Y]$ must be constant on the atoms of $\sigma(Y)$. Since the atoms are $\{\omega_1, \omega_2\}$ and $\{\omega_3, \omega_4\}$, the value of the expectation can only change when $Y$ changes.

In [2]:
import numpy as np
import pandas as pd

# Define the sample space data
data = {
    'outcome': ['w1', 'w2', 'w3', 'w4'],
    'prob': [0.5, 0.1, 0.2, 0.2],
    'X': [0, 1, 0, 1],  # Fraud indicator
    'Y': [0, 0, 1, 1]   # Risk class
}
df = pd.DataFrame(data)

# 4. Compute E[X|Y]

def get_cond_exp(y_val):
    subset = df[df['Y'] == y_val]
    return (subset['X'] * subset['prob']).sum() / subset['prob'].sum()

e_x_y0 = get_cond_exp(0)
e_x_y1 = get_cond_exp(1)

print(f"E[X|Y=0] = {e_x_y0:.4f} (expected 1/6)")
print(f"E[X|Y=1] = {e_x_y1:.4f} (expected 1/2)")

# Map E[X|Y] back to the outcomes for MSE calculation
df['E_X_Y'] = df['Y'].map({0: e_x_y0, 1: e_x_y1})

# 8. Calculate MSE: E[(X - E[X|Y])^2]
df['sq_error'] = (df['X'] - df['E_X_Y'])**2
mse_with_y = (df['sq_error'] * df['prob']).sum()

# 9. MSE without Y: E[(X - E[X])^2] = Var(X)
e_x = (df['X'] * df['prob']).sum()
mse_no_y = ((df['X'] - e_x)**2 * df['prob']).sum()

print(f"\nMSE with Y: {mse_with_y:.4f}")
print(f"MSE without Y (Var(X)): {mse_no_y:.4f}")
print(f"Reduction in variance: {mse_no_y - mse_with_y:.4f} (This is Var(E[X|Y]))")

# 10. Information Theory
def entropy(probs):
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

# H(X)
p_x1 = df[df['X'] == 1]['prob'].sum()
p_x = np.array([1 - p_x1, p_x1])
h_x = entropy(p_x)

# H(X|Y) = P(Y=0)H(X|Y=0) + P(Y=1)H(X|Y=1)
p_y0 = df[df['Y'] == 0]['prob'].sum()
p_y1 = df[df['Y'] == 1]['prob'].sum()

# Conditional distributions
p_x_y0 = np.array([1 - e_x_y0, e_x_y0])
p_x_y1 = np.array([1 - e_x_y1, e_x_y1])
h_x_y = p_y0 * entropy(p_x_y0) + p_y1 * entropy(p_x_y1)

mi = h_x - h_x_y

print(f"\n--- Information Theory Results ---")
print(f"Entropy H(X): {h_x:.4f} bits")
print(f"Cond. Entropy H(X|Y): {h_x_y:.4f} bits")
print(f"Mutual Information I(X;Y): {mi:.4f} bits")
print(f"Uncertainty removed: {(mi/h_x)*100:.2f}%")

E[X|Y=0] = 0.1667 (expected 1/6)
E[X|Y=1] = 0.5000 (expected 1/2)

MSE with Y: 0.1833
MSE without Y (Var(X)): 0.2100
Reduction in variance: 0.0267 (This is Var(E[X|Y]))

--- Information Theory Results ---
Entropy H(X): 0.8813 bits
Cond. Entropy H(X|Y): 0.7900 bits
Mutual Information I(X;Y): 0.0913 bits
Uncertainty removed: 10.36%
